# LangChain SQL Database Loader

LangChain provides tools to interact with SQL databases, enabling seamless integration of database queries into language model workflows. The SQL Database Loader is a utility that allows you to load data from SQL databases into LangChain for further processing, analysis, or chaining with other components.

## Types of SQL Database Loaders

LangChain supports several types of SQL database loaders, each designed for different use cases:

### 1. **Standard SQLDatabase Loader**
- **Purpose:** Connects to a SQL database (e.g., SQLite, PostgreSQL, MySQL) and enables querying tables directly.
- **Features:** 
    - Executes SQL queries.
    - Retrieves results as pandas DataFrames or dictionaries.
    - Supports schema introspection.
- **Usage Example:**
    ```python
    from langchain.sql_database import SQLDatabase

    db = SQLDatabase.from_uri("sqlite:///data/database_files/company_data.db")
    results = db.run("SELECT * FROM employees")
    ```

### 2. **Custom SQLDatabase Loaders**
- **Purpose:** Extend or customize the standard loader to handle specific requirements, such as:
    - Custom authentication.
    - Query optimization.
    - Data transformation before loading.
    - Integration with non-standard SQL dialects.
- **Implementation:** Subclass the base loader and override methods like `run`, `get_table_names`, or `get_table_info`.
- **Usage Example:**
    ```python
    from langchain.sql_database import SQLDatabase

    class MyCustomLoader(SQLDatabase):
            def run(self, query):
                    # Custom logic before running query
                    print("Running custom query:", query)
                    return super().run(query)
    ```

### 3. **Async SQLDatabase Loader**
- **Purpose:** For asynchronous workflows, enabling non-blocking database operations.
- **Features:** Uses async database drivers and methods.
- **Usage Example:**
    ```python
    from langchain.sql_database import AsyncSQLDatabase

    async_db = AsyncSQLDatabase.from_uri("sqlite+aiosqlite:///data/database_files/company_data.db")
    results = await async_db.run("SELECT * FROM employees")
    ```

## Comparison: Standard vs Custom SQLDatabase Loaders

| Feature                | Standard Loader         | Custom Loader                |
|------------------------|------------------------|------------------------------|
| **Ease of Use**        | Plug-and-play          | Requires subclassing         |
| **Flexibility**        | Limited to defaults    | Highly customizable          |
| **Performance Tuning** | Basic                  | Can optimize queries         |
| **Data Transformation**| Minimal                | Pre/post-processing possible |
| **Integration**        | Standard SQL dialects  | Non-standard dialects        |

## Summary

- **Standard loaders** are ideal for quick integration and basic querying.
- **Custom loaders** are recommended for advanced use cases, such as handling complex schemas, optimizing performance, or integrating with proprietary databases.
- **Async loaders** are useful for scalable, non-blocking applications.

LangChain’s SQLDatabase loader ecosystem is designed to be extensible, allowing developers to tailor database interactions to their specific needs.

In [1]:
import os
import sqlite3
import json

os.makedirs("data/database_files", exist_ok=True)

In [2]:
# Create sample database and table
conn = sqlite3.connect("data/database_files/company_data.db")
cursor = conn.cursor()

# Create an employees table if it doesn't exist, drop if it exists to avoid errors
cursor.execute('DROP TABLE IF EXISTS employees')
cursor.execute('''
CREATE TABLE IF NOT EXISTS employees (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    position TEXT NOT NULL,
    skills TEXT NOT NULL,
    address TEXT NOT NULL
)
''')
conn.commit()

In [3]:
# Create a project table which has a foreign key relationship with employees table, drop the table and recreate to avoid errors
cursor.execute('DROP TABLE IF EXISTS projects')
cursor.execute('''
CREATE TABLE IF NOT EXISTS projects (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    description TEXT NOT NULL,
    employee_id INTEGER,
    FOREIGN KEY (employee_id) REFERENCES employees (id)
)
''')
conn.commit()
# Insert sample data into employees table
employees = [
    (1001, "Alice", "Software Engineer", "Python, Java", "123 Main St, Anytown, USA"),
    (1002, "Bob", "Data Scientist", "Python, R", "456 Elm St, Othertown, USA"),
    (1003, "Charlie", "Product Manager", "Management, Agile", "789 Oak St, Sometown, USA"),
    (1004, "David", "DevOps Engineer", "AWS, Docker", "321 Pine St, Anycity, USA"),
    (1005, "Eve", "UX Designer", "Figma, Adobe XD", "654 Maple St, Othercity, USA")
]
cursor.executemany('''
INSERT INTO employees (id, name, position, skills, address) VALUES (?, ?, ?, ?, ?)
''', employees)
conn.commit()
# Insert sample data into projects table
projects = [
    (1, "Project Alpha", "Description for Project Alpha", 1001),
    (2, "Project Beta", "Description for Project Beta", 1002),
    (3, "Project Gamma", "Description for Project Gamma", 1003),
    (4, "Project Delta", "Description for Project Delta", 1004),
    (5, "Project Epsilon", "Description for Project Epsilon", 1005)
]
cursor.executemany('''
INSERT INTO projects (id, name, description, employee_id) VALUES (?, ?, ?, ?)
''', projects)
conn.commit()

In [4]:
cursor.execute('SELECT * FROM employees')

**Standard SQLDatabase Loader Example :**

In [7]:
from langchain_community.utilities import SQLDatabase
from langchain_community.document_loaders import SQLDatabaseLoader

db = SQLDatabase.from_uri("sqlite:///data/database_files/company_data.db")
print(f"Tables in the database: {db.get_usable_table_names()}") 
print(f"Table DDLs:\n{db.get_table_info()}")


Tables in the database: ['employees', 'projects']
Table DDLs:

CREATE TABLE employees (
	id INTEGER, 
	name TEXT NOT NULL, 
	position TEXT NOT NULL, 
	skills TEXT NOT NULL, 
	address TEXT NOT NULL, 
	PRIMARY KEY (id)
)

/*
3 rows from employees table:
id	name	position	skills	address
1001	Alice	Software Engineer	Python, Java	123 Main St, Anytown, USA
1002	Bob	Data Scientist	Python, R	456 Elm St, Othertown, USA
1003	Charlie	Product Manager	Management, Agile	789 Oak St, Sometown, USA
*/


CREATE TABLE projects (
	id INTEGER, 
	name TEXT NOT NULL, 
	description TEXT NOT NULL, 
	employee_id INTEGER, 
	PRIMARY KEY (id), 
	FOREIGN KEY(employee_id) REFERENCES employees (id)
)

/*
3 rows from projects table:
id	name	description	employee_id
1	Project Alpha	Description for Project Alpha	1001
2	Project Beta	Description for Project Beta	1002
3	Project Gamma	Description for Project Gamma	1003
*/


In [ ]:
# **Standard SQLDatabase Loader Example :**
db = SQLDatabase.from_uri("sqlite:///data/database_files/company_data.db")
query = "SELECT * FROM employees"
loader = SQLDatabaseLoader(db=db, query=query )
docs = loader.load()
print(f"Loaded {len(docs)} documents from employees table")
docs

Loaded 5 documents from employees table


[Document(metadata={}, page_content='id: 1001\nname: Alice\nposition: Software Engineer\nskills: Python, Java\naddress: 123 Main St, Anytown, USA'),
 Document(metadata={}, page_content='id: 1002\nname: Bob\nposition: Data Scientist\nskills: Python, R\naddress: 456 Elm St, Othertown, USA'),
 Document(metadata={}, page_content='id: 1003\nname: Charlie\nposition: Product Manager\nskills: Management, Agile\naddress: 789 Oak St, Sometown, USA'),
 Document(metadata={}, page_content='id: 1004\nname: David\nposition: DevOps Engineer\nskills: AWS, Docker\naddress: 321 Pine St, Anycity, USA'),
 Document(metadata={}, page_content='id: 1005\nname: Eve\nposition: UX Designer\nskills: Figma, Adobe XD\naddress: 654 Maple St, Othercity, USA')]

**Custom SQLDatabaseLoader Example :**

In [23]:
from langchain_community.document_loaders import SQLDatabaseLoader
from langchain_core.documents import Document

class CustomSQLDatabaseLoader(SQLDatabaseLoader):
    def __init__(self, db_path: str):
        self.db_path = db_path
        self.db_uri = f"sqlite:///{db_path}"

    def load(self):
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute("SELECT name from sqlite_master WHERE type='table';")
        tables = cursor.fetchall()
        documents = []
        for table_name in tables:
            table = table_name[0]
            # get table schema
            cursor.execute(f"PRAGMA table_info({table});")
            columns = cursor.fetchall()
            # print(f"Schema for table {table}: {columns}")
            column_names = [col[1] for col in columns]
            cursor.execute(f"SELECT * FROM {table};")
            rows = cursor.fetchall()

            table_content = f"Table: {table}\n"
            table_content += f"Columns: {', '.join(column_names)}\n"
            table_content += f"Total Rows: {len(rows)}\n"
            table_content += "Sample Records:\n"
            for row in rows[:5]:  # Show only the first 5 rows as sample
                table_content += f"  {dict(zip(column_names, row))}\n"

            metadata = {
                "source": self.db_path,
                "table_name": table,
                "columns": column_names,
                "row_count": len(rows),
                "data_Type": "SQL_table"
            }
            documents.append(Document(page_content=table_content, metadata=metadata))

            # Create relationships based documents
            cursor.execute("""select e.name as employee_name, e.position as employee_position, p.name as project_name
                              from employees e
                              join projects p on e.id = p.employee_id;""")

            relationship_rows = cursor.fetchall()
            rel_content = f"Employee-Project Relationships:\n"
            rel_content += f"Total Relationships: {len(relationship_rows)}\n"
            for row in relationship_rows:
                employee_name, employee_position, project_name = row
                relationship_content = f"Employee: {employee_name}, Position: {employee_position}, Project: {project_name}\n"
                metadata = {
                    "source": self.db_path,
                    "relationship": "employee_project",
                    "employee_name": employee_name,
                    "project_name": project_name
                }
                documents.append(Document(page_content=relationship_content, metadata=metadata))

        conn.close()
        return documents

In [24]:
sql_loader = CustomSQLDatabaseLoader("data/database_files/company_data.db")
custom_sql_docs = sql_loader.load()
print(f"Loaded {len(custom_sql_docs)} custom SQL documents from function")
custom_sql_docs

Loaded 12 custom SQL documents from function


[Document(metadata={'source': 'data/database_files/company_data.db', 'table_name': 'employees', 'columns': ['id', 'name', 'position', 'skills', 'address'], 'row_count': 5, 'data_Type': 'SQL_table'}, page_content="Table: employees\nColumns: id, name, position, skills, address\nTotal Rows: 5\nSample Records:\n  {'id': 1001, 'name': 'Alice', 'position': 'Software Engineer', 'skills': 'Python, Java', 'address': '123 Main St, Anytown, USA'}\n  {'id': 1002, 'name': 'Bob', 'position': 'Data Scientist', 'skills': 'Python, R', 'address': '456 Elm St, Othertown, USA'}\n  {'id': 1003, 'name': 'Charlie', 'position': 'Product Manager', 'skills': 'Management, Agile', 'address': '789 Oak St, Sometown, USA'}\n  {'id': 1004, 'name': 'David', 'position': 'DevOps Engineer', 'skills': 'AWS, Docker', 'address': '321 Pine St, Anycity, USA'}\n  {'id': 1005, 'name': 'Eve', 'position': 'UX Designer', 'skills': 'Figma, Adobe XD', 'address': '654 Maple St, Othercity, USA'}\n"),
 Document(metadata={'source': 'dat